# Step 1 — Data Extraction & Financial Fact Store

Turn a ticker + quarter into a typed **Financial Fact Store** where every number carries
provenance — the grounding backbone (`docs/grounding.md`). Data is real **defeatbeta-api** data:
live when `USE_MOCK_DATA=false`, or cached real data from `mock/data/` offline (built by
`mock/build_mock_data.py`). Persistence is **SQLite** (stdlib — no parquet/duckdb).

In [1]:
import sys
from pathlib import Path
def _root():
    p = Path.cwd()
    for d in (p, *p.parents):
        if (d / "requirements.txt").exists():
            return d
    return p
ROOT = _root(); sys.path.insert(0, str(ROOT / "src"))
from ir_copilot.config import settings
print("ticker:", settings.ticker, "| period:", settings.period, "| data:",
      "mock(cached real)" if settings.use_mock_data else "live defeatbeta-api")

ticker: NVDA | period: FY2026Q2 | data: mock(cached real)


## Build the Fact Store
`build_fact_store` returns grounded `FinancialFact`s (value + unit + source + as-of). Offline it
reads cached real defeatbeta-api values; live it calls the API (Python 3.11+).

In [2]:
from ir_copilot.facts import build_fact_store, render_slots, ungrounded_numbers, REQUIRED_METRICS

store = build_fact_store(settings.ticker, settings.period, use_mock=settings.use_mock_data)
print(f"{len(store.facts)} facts, {len(store.gaps)} gaps")
import pandas as pd
df = store.to_dataframe()
df[["fact_id", "metric", "value", "unit", "source", "as_of"]]

15 facts, 0 gaps


,fact_id,metric,value,unit,source,as_of
0,F-0001,ttm_eps,6.529900e+00,USD,defeatbeta-api:ttm_eps,2026-04-30
1,F-0002,ttm_pe,3.188000e+01,x,defeatbeta-api:ttm_pe,2026-06-09
2,F-0003,market_cap,5.042570e+12,USD,defeatbeta-api:market_capitalization,2026-06-09
3,F-0004,ps_ratio,1.989000e+01,x,defeatbeta-api:ps_ratio,2026-06-09
4,F-0005,pb_ratio,2.580000e+01,x,defeatbeta-api:pb_ratio,2026-06-09
5,F-0006,peg_ratio,1.500000e-01,x,defeatbeta-api:peg_ratio,2026-06-09
6,F-0007,roe,3.306000e+01,%,defeatbeta-api:roe,2026-04-30
7,F-0008,roa,2.502000e+01,%,defeatbeta-api:roa,2026-04-30
8,F-0009,roic,3.143000e+01,%,defeatbeta-api:roic,2026-04-30
9,F-0010,wacc,2.270000e+01,%,defeatbeta-api:wacc,2026-06-09


## Persist to SQLite + reload
The relational `facts` table makes grounded financials queryable with plain SQL; a JSON copy is
written for inspection. Artifacts live under the git-ignored `artifacts/` dir.

In [ ]:
from ir_copilot.cache import Cache

cache = Cache()
n = cache.save_facts(store)
json_path = store.save_json(settings.artifacts_dir / "fact_store")
print(f"saved {n} facts to sqlite ({cache.path.name}) + json ({json_path.name})")

reloaded = cache.load_facts(settings.ticker, settings.period)
print(f"reloaded {len(reloaded.facts)} facts from sqlite; revenue matches:",
      reloaded.value("revenue") == store.value("revenue"))

## Grounding guard + coverage check
Every number in generated text must trace to a fact (`$bn`/`$tn` and percent forms allowed);
every required metric must be present. These are what the verifier enforces downstream.

In [ ]:
eps = store.slot("ttm_eps")                       # -> "{{F-00xx}}"
grounded = render_slots(f"TTM EPS was {eps}.", store)   # -> real value substituted
hallucinated = "TTM EPS was 9.99 and gross margin was 250.0%."

print("grounded sentence :", grounded)
print("  ungrounded numbers:", ungrounded_numbers(grounded, store))
print("hallucinated       :", ungrounded_numbers(hallucinated, store))
assert ungrounded_numbers(grounded, store) == []
assert ungrounded_numbers(hallucinated, store)

present = {f.metric for f in store.facts}
missing = REQUIRED_METRICS - present
print("coverage missing:", missing or "none")
assert not missing
print("\nGrounding guard works: hallucinated numbers flagged, coverage complete.")

**Next (Step 2):** the multimodal Knowledge Wiki — Qdrant ingest + cited retrieval
(`02_wiki_ingestion_qdrant.ipynb`).